# PoneTTS on Colab

Runs the [PoneTTS](https://github.com/modChad/PoneTTS) server (MOSS-TTS-PNY) on a Colab GPU and opens a public link to it.

**Use:** set the options and server settings in step 1, then *Runtime → Run all*. Step 6 prints a URL. Running *Run all* again in the same session keeps the downloaded model.
- **Game (Home: Equestria):** AI Settings → Voice server **Local moss-tts**, paste the URL **with `/tts` on the end**, voice mode **Character voices via server**.
- **SillyTavern:** XTTS provider, URL = the printed address.
- **Web page:** open the printed URL in a browser.

**Keep this tab open while you use it.** When the session ends the URL stops working; *Run all* again for a new one.
The URL is unlisted, but anyone who has it can use your server. Don't post it.

## 1. Options and GPU check

In [ ]:
# --- edit these ---------------------------------------------------------------
REPO_URL    = "https://github.com/modChad/PoneTTS.git"   # the project (the folder that contains server/)
SERVER_DIR  = "server"         # folder inside the repo that holds server.py
PRESET      = "auto"           # auto = pick by GPU memory | quality (~14-16 GB, vocoder on the GPU) | balanced (~9-12 GB, vocoder on the CPU) | low_vram (~7-8 GB) | potato (~6 GB)
VOCODER     = ""               # "" = the preset's own. "onnx_cuda" = force the GPU vocoder (needs ~5 GB more VRAM), "onnx_cpu" = force the CPU one
COMPILE     = "default"        # Colab has ~2 CPU cores; "default" starts in minutes, the presets' own max-autotune takes 25+ min
FRESH_CLONE = False            # True = delete the earlier clone and its downloaded model, then clone and download again
# -------------------------------------------------------------------------------
import os, subprocess

if not REPO_URL.strip():
    raise SystemExit("Set REPO_URL at the top of this cell to the project's git URL, then Runtime → Run all.")

r = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits"], capture_output=True, text=True)
if r.returncode != 0 or not r.stdout.strip():
    raise SystemExit("No GPU. Runtime → Change runtime type → pick a GPU, then Runtime → Run all.")
_gpu, _mib = [s.strip() for s in r.stdout.strip().splitlines()[0].rsplit(",", 1)]
VRAM_MB = int(float(_mib))
VRAM_GB = VRAM_MB / 1024
print(f"GPU: {_gpu} ({VRAM_GB:.1f} GB)")

# same thresholds as run.sh
PRESETS = ("quality", "balanced", "low_vram", "potato")
if PRESET == "auto":
    PRESET = ("potato" if VRAM_MB < 7168 else "low_vram" if VRAM_MB <= 9216
              else "balanced" if VRAM_MB <= 12800 else "quality")
    print("preset (auto):", PRESET)
elif PRESET not in PRESETS:
    raise SystemExit(f"PRESET must be auto, {', '.join(PRESETS)}; got {PRESET!r}")

_need = {"quality": 12.5, "balanced": 9, "low_vram": 7, "potato": 5.5}[PRESET]
if VOCODER == "onnx_cuda" and PRESET != "quality":
    _need += 5
if VRAM_GB < _need:
    print(f"WARNING: preset {PRESET!r}{' + GPU vocoder' if VOCODER == 'onnx_cuda' else ''} wants about {_need} GB, "
          f"this GPU has {VRAM_GB:.1f} GB. It may run out of memory; set PRESET = \"auto\" or a smaller preset.")

In [ ]:
#@title Server settings
#@markdown Read when the server starts in step 5. Most can also be changed live from the web page's **Settings** panel, which wins over this form until you press *Reset* there.
#@markdown To change one here later, edit it, run this cell, then run step 5 again (the server restarts and warms up again).
#@markdown ---
#@markdown **Voice**
AUDIO_TEMPERATURE = 0.85 #@param {type:"number"}
#@markdown Default sampling temperature. Lower is steadier, higher is more expressive. A request's own `audio_temperature` wins.

#@markdown **Pauses** (seconds)
SENTENCE_PAUSE_SEC = 0.12 #@param {type:"number"}
COMMA_PAUSE_SEC = 0.08 #@param {type:"number"}
PAUSE_SEC = 0.35 #@param {type:"number"}
#@markdown `PAUSE_SEC` is the silence at an em-dash or ellipsis.
COMMA_SPLIT_MIN_WORDS = 20 #@param {type:"integer"}
#@markdown Sentences longer than this are also split at commas. `0` turns comma splitting off.

#@markdown **Streaming** (SillyTavern's XTTSv2 streaming)
STREAM_BUFFER_SEC = 3 #@param {type:"number"}
#@markdown Seconds of audio generated before playback starts. `0` = off, large = wait for the whole line.

#@markdown **Quality checks** (Whisper runs on Colab's slow CPU, so these cost time)
FIX_STUCK_WORDS = False #@param {type:"boolean"}
#@markdown Regenerate sentences where a word gets stuck droning.
STUCK_WHISPER_GATE = 1.3 #@param {type:"number"}
#@markdown Whisper only listens to sentences this many times longer than expected. Lower catches more; `0` listens to every sentence (slowest).
LOOP_GUARD_REL_MULT = 1.6 #@param {type:"number"}
LOOP_GUARD_MIN_HEARD = 0.5 #@param {type:"number"}
FAILED_CLAUSE_RETRIES = 2 #@param {type:"integer"}
#@markdown A sentence this many times longer than expected gets a repeat check; if Whisper hears less than `LOOP_GUARD_MIN_HEARD` of its words it's regenerated up to `FAILED_CLAUSE_RETRIES` times.

#@markdown **Pronunciation helpers** (`pronunciations.txt` always applies)
FIX_PRON = False #@param {type:"boolean"}
#@markdown Check rare words with Whisper and fix their pronunciation (slower).
AUTO_RESPELL = False #@param {type:"boolean"}
#@markdown Respell rare words phonetically before speaking them.

#@markdown **Other**
HISTORY_MAX = 500 #@param {type:"integer"}
#@markdown Generated lines kept in the history. `0` turns history off.
SFX_ALLOW_NETWORK = False #@param {type:"boolean"}
#@markdown Let `/sfx` fetch sound effects from Freesound / the BBC archive. Off = cache only.
FREESOUND_API_KEY = "" #@param {type:"string"}
#@markdown Optional. Better Freesound results with a free key from freesound.org/apiv2/apply.

#@markdown **Admin key** for the web page's Settings panel
ADMIN_KEY = "" #@param {type:"string"}
#@markdown Leave empty to generate one (printed with the link in step 6). Anyone with the public link could otherwise change your settings. Speech requests never need it.
import re as _re, secrets as _secrets
ADMIN_KEY = ADMIN_KEY.strip() or globals().get("_GENERATED_ADMIN_KEY") or _secrets.token_urlsafe(18)
if not _re.fullmatch(r"[A-Za-z0-9._-]{8,}", ADMIN_KEY):
    raise SystemExit("ADMIN_KEY: use at least 8 letters, digits, dots, underscores or hyphens (or leave it empty).")
_GENERATED_ADMIN_KEY = ADMIN_KEY   # keep the same generated key when this cell is run again

_b = lambda v: "1" if v else "0"
SERVER_ENV = {
    "MOSS_AUDIO_TEMPERATURE": str(AUDIO_TEMPERATURE),
    "MOSS_SENTENCE_PAUSE_SEC": str(SENTENCE_PAUSE_SEC),
    "MOSS_COMMA_PAUSE_SEC": str(COMMA_PAUSE_SEC),
    "MOSS_PAUSE_SEC": str(PAUSE_SEC),
    "MOSS_COMMA_SPLIT_MIN_WORDS": str(int(COMMA_SPLIT_MIN_WORDS)),
    "MOSS_STREAM_BUFFER_SEC": str(STREAM_BUFFER_SEC),
    "MOSS_FIX_STUCK_WORDS": _b(FIX_STUCK_WORDS),
    "MOSS_STUCK_WHISPER_GATE": str(STUCK_WHISPER_GATE),
    "MOSS_LOOP_GUARD_REL_MULT": str(LOOP_GUARD_REL_MULT),
    "MOSS_LOOP_GUARD_MIN_HEARD": str(LOOP_GUARD_MIN_HEARD),
    "MOSS_FAILED_CLAUSE_RETRIES": str(int(FAILED_CLAUSE_RETRIES)),
    "MOSS_FIX_PRON": _b(FIX_PRON),
    "MOSS_AUTO_RESPELL": _b(AUTO_RESPELL),
    "MOSS_HISTORY_MAX": str(int(HISTORY_MAX)),
    "MOSS_SFX_ALLOW_NETWORK": _b(SFX_ALLOW_NETWORK),
    **({"FREESOUND_API_KEY": FREESOUND_API_KEY.strip()} if FREESOUND_API_KEY.strip() else {}),
    "MOSS_ADMIN_KEY": ADMIN_KEY,
}
print("server settings:", ", ".join(f"{k.replace('MOSS_', '')}={v}" for k, v in SERVER_ENV.items()
                                    if k not in ("FREESOUND_API_KEY", "MOSS_ADMIN_KEY")))

## 2. Get the project

In [ ]:
import os, subprocess
os.environ["REPO_URL"] = REPO_URL
_origin = subprocess.run(["git", "-C", "/content/pone", "remote", "get-url", "origin"], capture_output=True, text=True).stdout.strip()
if _origin == REPO_URL and not FRESH_CLONE:
    print("Keeping the project cloned earlier this session (and any model it already downloaded).")
    print("Set FRESH_CLONE = True in step 1 to start over.")
else:
    subprocess.run('cd /content && rm -rf pone && git clone --depth 1 "$REPO_URL" pone', shell=True, check=True)
APP = f"/content/pone/{SERVER_DIR}"
assert os.path.exists(f"{APP}/server.py"), f"server.py not found in {APP}"
assert os.path.exists(f"{APP}/prepare.py"), f"prepare.py not found in {APP}: this notebook needs a newer copy of the project"
print("project ok:", APP)

## 3. Install (about 10 minutes)

In [ ]:
%%bash
set -e
cd /content
pip install -q uv
[ -x venv/bin/python ] || uv venv -q venv --python 3.12
uv pip install -q -p venv/bin/python --index-url https://download.pytorch.org/whl/cu128 torch==2.11.0 torchaudio==2.11.0

In [ ]:
# requirements come from the project itself, so they are always the version it was tested with
r = subprocess.run(f"uv pip install -q -p /content/venv/bin/python -r {APP}/requirements.txt", shell=True)
assert r.returncode == 0, "requirements failed to install; the message above says which package"

# faster-whisper pulls in the CPU-only "onnxruntime", which overwrites onnxruntime-gpu and removes its CUDA provider
# (the vocoder then silently runs on Colab's 2 CPU cores). Put the GPU build back, at the version requirements.txt pins.
import re
_pin = re.search(r"onnxruntime-gpu==([0-9.]+)", open(f"{APP}/requirements.txt").read())
_ver = _pin.group(1) if _pin else "1.23.2"
subprocess.run("uv pip uninstall -q -p /content/venv/bin/python onnxruntime onnxruntime-gpu", shell=True)
r = subprocess.run(f"uv pip install -q -p /content/venv/bin/python --no-deps onnxruntime-gpu=={_ver}", shell=True)
assert r.returncode == 0, "could not reinstall onnxruntime-gpu"
out = subprocess.run(["/content/venv/bin/python", "-c", "import onnxruntime as o; print(o.get_available_providers())"], capture_output=True, text=True)
print("onnxruntime providers:", out.stdout.strip() or out.stderr[-300:])
assert "CUDAExecutionProvider" in out.stdout, "onnxruntime still has no CUDA provider"
print(subprocess.run(["/content/venv/bin/python", "-c", "import torch; print('torch', torch.__version__, '| cuda', torch.cuda.is_available())"], capture_output=True, text=True).stdout)

## 4. Download the model and apply the patches (about 15 minutes, 13 GB; skipped when re-running in the same session)
The voice patch (~100 MB) is downloaded too.

In [ ]:
import ast, json, time
from tqdm.auto import tqdm

# the plain HuggingFace downloader (download_model.py's default): the fast ones can run Colab out of RAM
env = dict(os.environ, HF_HUB_DISABLE_TELEMETRY="1")
MODEL_DIR = f"{APP}/release"
DL_MARK   = f"{MODEL_DIR}/.setup_done"            # prepare.py's marker: written after a full download
HF_CACHE  = os.path.expanduser("~/.cache/huggingface/hub")
ATTEMPTS  = 3                                      # downloads resume, so a retry continues where it stopped
BIG       = 10 * 1024**2                           # files under 10 MB share one bar

def _model_files():
    # [(name, bytes, etag)] for the files download_model.py fetches (its REPO_ID / PATTERNS)
    consts = {}
    for n in ast.walk(ast.parse(open(f"{APP}/download_model.py").read())):
        if isinstance(n, ast.Assign) and len(n.targets) == 1 and isinstance(n.targets[0], ast.Name) and n.targets[0].id in ("REPO_ID", "PATTERNS"):
            try: consts[n.targets[0].id] = ast.literal_eval(n.value)
            except Exception: pass
    pats = consts.get("PATTERNS") or []
    if isinstance(pats, str): pats = [pats]
    q = ("import json,fnmatch,sys\nfrom huggingface_hub import HfApi\n"
         "repo,pats=json.loads(sys.argv[1])\nout=[]\n"
         "for f in HfApi().model_info(repo,files_metadata=True).siblings:\n"
         "    if pats and not any(fnmatch.fnmatch(f.rfilename,p) for p in pats): continue\n"
         "    l=f.lfs; sha=(l.get('sha256') if isinstance(l,dict) else getattr(l,'sha256',None)) if l else None\n"
         "    out.append([f.rfilename, f.size or 0, sha or f.blob_id or ''])\n"
         "print(json.dumps(out))")
    o = subprocess.run(["/content/venv/bin/python", "-c", q, json.dumps([consts["REPO_ID"], list(pats)])],
                       capture_output=True, text=True, timeout=60)
    return [tuple(f) for f in json.loads(o.stdout)] if o.returncode == 0 else []

def _written(path):
    # bytes really on disk (a pre-sized file would otherwise jump to 100% at once)
    try:
        st = os.stat(path)
        return min(st.st_size, st.st_blocks * 512)
    except OSError:
        return 0

def _walk(*dirs):
    for d in dirs:
        for root, _sub, names in os.walk(d):
            for n in names:
                p = os.path.join(root, n)
                yield os.path.relpath(p, d).replace(os.sep, "/"), n, p

def _per_file(files):
    # finished files by name, partial ones (*.incomplete) by etag; works for local_dir and the HF cache
    by_name = {f[0] for f in files}
    by_etag = {f[2]: f[0] for f in files if f[2]}
    got = {}
    for rel, name, p in _walk(MODEL_DIR, HF_CACHE):
        if name.endswith(".incomplete"):
            stem = name[:-len(".incomplete")]
            key = by_etag.get(stem) or by_etag.get(stem.rsplit(".", 1)[-1])
        else:
            key = (rel if rel in by_name else None) or by_etag.get(name) or next((fn for fn in by_name if rel.endswith("/" + fn)), None)
        if key: got[key] = max(got.get(key, 0), _written(p))
    return got

if os.path.exists(DL_MARK):
    print("model already downloaded this session, skipping the download")
else:
    try:
        FILES = _model_files()
    except Exception as e:
        FILES = []; print("could not list the model files:", e)
    if FILES: print(f"downloading {len(FILES)} files, {sum(f[1] for f in FILES)/1024**3:.1f} GB")
    else:     print("showing one bar for the whole download (about 13 GB)")

    kw = dict(unit="B", unit_scale=True, unit_divisor=1024, leave=True)
    short = lambda s: s if len(s) <= 40 else "…" + s[-39:]
    sizes = {f[0]: f[1] for f in FILES}
    bars, group = {}, {}
    for fn, size, _e in sorted(FILES):
        if size >= BIG:
            bars[fn] = tqdm(total=size, desc=short(fn), **kw); group[fn] = fn
    small = [f for f in FILES if f[1] < BIG]
    if small:
        bars["small"] = tqdm(total=sum(f[1] for f in small), desc=f"{len(small)} small files", **kw)
        for f in small: group[f[0]] = "small"
    if not FILES:
        bars["all"] = tqdm(total=13 * 1024**3, desc="model", **kw)

    def _refresh():
        if FILES:
            now = {}
            for fn, b in _per_file(FILES).items():
                now[group[fn]] = now.get(group[fn], 0) + min(b, sizes[fn])
        else:
            now = {"all": sum(_written(p) for _r, _n, p in _walk(MODEL_DIR))}
        for k, bar in bars.items():
            v = min(now.get(k, 0), bar.total)
            if v > bar.n: bar.update(v - bar.n)
        return sum(now.values())

    dl_log = open("/content/download.log", "a")
    t0 = time.time()
    for attempt in range(1, ATTEMPTS + 1):
        proc = subprocess.Popen(["/content/venv/bin/python", "download_model.py"], cwd=APP, env=env, stdout=dl_log, stderr=subprocess.STDOUT)
        last_total, last_growth, last_warn = -1, time.time(), 0
        while proc.poll() is None:
            time.sleep(2)
            total = _refresh()
            if total > last_total:
                last_total, last_growth = total, time.time()
            elif time.time() - last_growth > 120 and time.time() - last_warn > 120:
                tqdm.write(f"no progress for {time.time()-last_growth:.0f} s, still waiting (log: /content/download.log)")
                last_warn = time.time()
        if proc.returncode == 0: break
        tqdm.write(f"download attempt {attempt} of {ATTEMPTS} failed (exit {proc.returncode})" + (", retrying..." if attempt < ATTEMPTS else ""))
    dl_log.close()
    if proc.returncode != 0:
        for bar in bars.values(): bar.close()
        print(open("/content/download.log", errors="replace").read()[-3000:])
        raise SystemExit("model download failed; the end of its log is above")
    for bar in bars.values():
        if bar.n < bar.total: bar.update(bar.total - bar.n)
        bar.close()
    open(DL_MARK, "w").close()
    print(f"download finished in {time.time()-t0:.0f} s")

# --- voice patch + bundle fix: the same prepare.py the Docker image and setup scripts use.
# It skips the download (marker above) and anything already applied.
r = subprocess.run(["/content/venv/bin/python", "prepare.py"], cwd=APP, env=env)
if r.returncode != 0:
    raise SystemExit(f"prepare.py failed (exit {r.returncode}); see the output above")
print("model ready")

## 5. Start the server (warm-up)

In [ ]:
import re, time, glob, site
from pathlib import Path

# stop a server left over from an earlier run of this step
_old = globals().get("server")
if isinstance(_old, subprocess.Popen) and _old.poll() is None:
    _old.terminate()
    try: _old.wait(15)
    except subprocess.TimeoutExpired: _old.kill()

# let onnxruntime find cuDNN/cuBLAS from the venv's nvidia-* packages (what entrypoint.sh does)
nv = [str(d) for d in glob.glob("/content/venv/lib/python3.12/site-packages/nvidia/*/lib") if os.path.isdir(d)]
env = dict(os.environ, PYTHONUNBUFFERED="1", HF_HUB_DISABLE_TELEMETRY="1",
           MOSS_PERF_PRESET=PRESET, MOSS_COMPILE_MODE=COMPILE, **({"MOSS_DECODER_RUNTIME": VOCODER} if VOCODER else {}),
           **SERVER_ENV,   # the "Server settings" form in step 1
           MOSS_TTS_NVIDIA_LD_LIBRARY_PATH_READY="1",
           LD_LIBRARY_PATH=":".join(nv + ([os.environ["LD_LIBRARY_PATH"]] if os.environ.get("LD_LIBRARY_PATH") else [])))
PORT = 8800
log_path = "/content/server.log"
server = subprocess.Popen(["/content/venv/bin/python", "-u", "server.py", "--host", "127.0.0.1", "--port", str(PORT), "--preset", PRESET],
                          cwd=APP, env=env, stdout=open(log_path, "w"), stderr=subprocess.STDOUT)
LIMIT_S, start = 45 * 60, time.time()
fmt = lambda sec: f"{int(sec) // 60}m {int(sec) % 60:02d}s"
STAGES = {   # friendly line printed when the server reaches each stage
    "[server] loading PoneTTS": "Loading the model onto the GPU...",
    "[server] warming up": "Warming up: compiling GPU kernels. This is the slow part (a few minutes).",
    "[server] ready": "Model ready. Opening the port...",
}
print(f"Starting the server (preset {PRESET}). The first start usually takes 3-10 minutes on Colab.", flush=True)
import json, urllib.request
done_lines, last_print = 0, time.time()
while True:
    lines = open(log_path, errors="replace").read().replace("\r", "\n").split("\n")
    for line in lines[done_lines:-1]:          # complete lines only
        line = line.strip()
        if not line.startswith("["):           # the server's own tagged lines; skip library noise
            continue
        stage = next((msg for key, msg in STAGES.items() if line.startswith(key)), None)
        if stage:
            print(f"{fmt(time.time() - start):>8}  {stage}", flush=True)
        print(f"{'':>8}    {line[:150]}", flush=True)
        last_print = time.time()
    done_lines = max(done_lines, len(lines) - 1)
    try:   # ready = the HTTP server answers, not just the model loaded
        if json.loads(urllib.request.urlopen(f"http://127.0.0.1:{PORT}/health", timeout=3).read()).get("status") == "ok":
            break
    except Exception:
        pass
    if server.poll() is not None:
        print(open(log_path, errors="replace").read()[-4000:]); raise SystemExit("The server stopped. The log is above.")
    if time.time() - start > LIMIT_S:
        print(open(log_path, errors="replace").read()[-4000:]); raise SystemExit("Not ready after 45 minutes. The log is above.")
    if time.time() - last_print >= 30:
        tail = next((l.strip() for l in reversed(lines) if l.strip()), "")
        print(f"{fmt(time.time() - start):>8}  still working...  {tail[:90]}", flush=True)
        last_print = time.time()
    time.sleep(3)
print(f"{fmt(time.time() - start):>8}  Server ready.", flush=True)

## 6. Open the public link

In [ ]:
import urllib.request
cf = "/content/cloudflared"
if not os.path.exists(cf):
    urllib.request.urlretrieve("https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", cf)
    os.chmod(cf, 0o755)
_oldt = globals().get("tunnel")
if isinstance(_oldt, subprocess.Popen) and _oldt.poll() is None: _oldt.terminate()
tunnel = subprocess.Popen([cf, "tunnel", "--no-autoupdate", "--url", f"http://127.0.0.1:{PORT}"],
                          stdout=open("/content/tunnel.log", "w"), stderr=subprocess.STDOUT)
public = None
for _ in range(60):
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("/content/tunnel.log", errors="replace").read())
    if m: public = m.group(0); break
    time.sleep(2)
if not public: raise SystemExit("No public link. Run this step again; the log is /content/tunnel.log")
print("Web page / SillyTavern :", public)
print("Home: Equestria        :", public + "/tts")
print("Settings admin key     :", ADMIN_KEY)

## 7. Optional: test line

In [ ]:
import json
from IPython.display import Audio, display
body = json.dumps({"text": "[happy] Hi Anon! The voice server is working.", "speaker_id": 31, "format": "wav"}).encode()
t0 = time.time()
req = urllib.request.Request(f"http://127.0.0.1:{PORT}/tts", data=body, headers={"Content-Type": "application/json", "X-Client-App": "colab-test"})
wav = urllib.request.urlopen(req, timeout=300).read()
print(f"took {time.time()-t0:.1f}s")
display(Audio(wav))

## 8. Optional: watch the server
Prints whether the server is answering, once a minute; stop with the ■ button.
This does **not** keep Colab awake: an idle tab can still be disconnected, and every session has a time limit. Check back on the tab now and then.

In [ ]:
try:
    while True:
        try:
            urllib.request.urlopen(f"http://127.0.0.1:{PORT}/health", timeout=5).read(1); status = "ok"
        except Exception as e:
            status = f"server not answering ({e})"
        print(time.strftime("%H:%M"), status, flush=True)
        time.sleep(60)
except KeyboardInterrupt:
    print("stopped watching (the server keeps running)")